# Scaling data

In [1]:
import pandas as pd
import numpy as np
from IPython.core.pylabtools import figsize
from PIL.ImageChops import difference
from matplotlib import pyplot as plt
from scipy.signal.windows import blackman
from scipy.spatial import distance
from statsmodels.graphics.tukeyplot import results
import os
import re
from utils import *

In [88]:
df_renamed = pd.read_excel("Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff.xlsx")

In [89]:
df_renamed
# df_test = df_renamed.head(10).copy()
# df_test

,protein_Id,protein_name,site,raw_abs_EGF_full_r1,raw_abs_EGF_full_r2,raw_abs_EGF_full_r3,raw_abs_EGF_full_r4,raw_abs_EGF_starve_r1,raw_abs_EGF_starve_r2,raw_abs_EGF_starve_r3,...,raw_CV_EGFnINS_full,raw_CV_EGFnINS_starve,raw_CV_EGFnINS_1,raw_CV_EGFnINS_2,raw_CV_EGFnINS_5,raw_CV_EGFnINS_10,raw_CV_EGFnINS_90,difference_EGF_vs_EGFnINS,difference_INS_vs_EGFnINS,difference_geometric_mean
0,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_0~SGSGNFGGGR,0.000000e+00,3.413601e+04,0.000000e+00,0.000000e+00,0.000000,11970.095745,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.961789,0.736883,0.841859
1,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_1_S154~SGsGNFGGGR,2.526291e+06,3.107422e+06,2.844213e+06,2.838165e+06,462877.845884,406460.086218,466606.366000,...,8.401287,12.205152,20.696008,15.790040,8.688666,20.424021,5.861373,1.801337,1.559497,1.676061
2,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_0~NQGGYGGSSSSSSYGSGR,2.538396e+05,2.912180e+05,1.878137e+05,2.302995e+05,370294.907420,351169.823470,391574.055743,...,17.990497,4.626026,9.199745,9.510235,11.659787,23.456764,9.822815,0.471713,0.414641,0.442258
3,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S264~NQGGYGGSsSSSSYGSGR,0.000000e+00,3.436628e+04,0.000000e+00,3.184211e+04,0.000000,36947.507319,0.000000,...,5.391637,17.556007,26.924308,19.852499,38.291377,48.538109,8.717235,1.186141,1.430206,1.302469
4,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S266~NQGGYGGSSSsSSYGSGR,4.004880e+05,5.070251e+05,2.948686e+05,0.000000e+00,732858.964004,703950.993495,706990.737660,...,26.467117,2.222977,7.708196,15.675311,15.772571,19.656534,13.218904,0.472098,0.296726,0.374278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34997,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_338_1_1_S338~SLsFEMQQDELIEK,3.612031e+04,3.523211e+04,2.328310e+04,2.882678e+04,28158.918559,31337.999945,38445.424548,...,19.468766,13.174961,3.261711,8.840158,11.748886,16.269093,4.782342,0.218582,0.496006,0.329269
34998,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_0~SLSFEMQQDELIEKPMSPMQYAR,0.000000e+00,0.000000e+00,0.000000e+00,1.989511e+04,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.721415,0.581098,0.647466
34999,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_1_S338~SLsFEMQQDELIEKPMSPMQYAR,2.252726e+05,2.548493e+05,1.697470e+05,1.780225e+05,202512.144807,165807.326109,237987.604392,...,19.428443,18.950531,10.566809,16.164894,9.825485,23.560264,12.615745,0.294116,0.153964,0.212798
35000,Q9Y6Y8,SEC23IP,Q9Y6Y8_118_138_1_0~PLTALPFTTGSQDVSNAFSPSISK,0.000000e+00,0.000000e+00,3.689021e+03,0.000000e+00,0.000000,0.000000,5253.875296,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.186760,0.806254,0.978177


In [90]:
column_names = df_renamed.columns.tolist()
column_selection = [element for element in column_names if 'log2_FC' in element]

column_selection = ["site"] + column_selection

new_columns = [name.replace("log2_FC_", "FC_scaled_") for name in column_selection]
subset_scaled = pd.DataFrame(columns=new_columns)

subset = df_renamed[column_selection]

for row in subset.itertuples():
    max_value = max(abs(n) for n in row[2:])# 0 is the index and 1 is the "site"
    if (max_value) == 0:
        print(row)
        continue

    scaled_list = [n / max_value for n in row[2:]]
    subset_scaled.loc[row[0]] =[row[1]] + scaled_list

subset_scaled

,site,FC_scaled_EGF_full,FC_scaled_EGF_starve,FC_scaled_EGF_1,FC_scaled_EGF_2,FC_scaled_EGF_5,FC_scaled_EGF_10,FC_scaled_EGF_90,FC_scaled_INS_full,FC_scaled_INS_starve,...,FC_scaled_INS_5,FC_scaled_INS_10,FC_scaled_INS_90,FC_scaled_EGFnINS_full,FC_scaled_EGFnINS_starve,FC_scaled_EGFnINS_1,FC_scaled_EGFnINS_2,FC_scaled_EGFnINS_5,FC_scaled_EGFnINS_10,FC_scaled_EGFnINS_90
0,A0A2R8Y4L2_152_154_1_0~SGSGNFGGGR,1.000000,0.0,-0.034121,-0.042529,-0.135560,0.220985,0.114021,1.000000,0.0,...,0.062765,-0.053044,0.392610,1.000000,0.0,0.009989,0.027808,0.150342,0.266855,0.384599
1,A0A2R8Y4L2_152_154_1_1_S154~SGsGNFGGGR,1.000000,0.0,-0.027497,0.018691,0.201352,0.389480,0.223468,1.000000,0.0,...,0.179028,0.290848,0.460393,1.000000,0.0,-0.013889,0.076752,0.324961,0.489716,0.582838
2,A0A2R8Y4L2_260_271_1_0~NQGGYGGSSSSSSYGSGR,-0.960321,0.0,-0.111209,-0.347658,-0.416773,-0.365559,-0.924598,-0.960321,0.0,...,-0.255660,-0.346432,-0.518686,-0.960321,0.0,-0.298015,-0.083473,-0.121019,-0.236009,-1.000000
3,A0A2R8Y4L2_260_271_1_1_S264~NQGGYGGSsSSSSYGSGR,-0.873135,0.0,0.258215,0.069255,-0.080050,-0.185280,-0.158402,-0.873135,0.0,...,-0.101023,-0.349209,-0.490396,-0.873135,0.0,-0.473961,0.472592,-0.426896,1.000000,-0.556594
4,A0A2R8Y4L2_260_271_1_1_S266~NQGGYGGSSSsSSYGSGR,-0.968252,0.0,-0.137996,-0.518940,-0.755894,-0.693874,-1.000000,-0.968252,0.0,...,-0.463319,-0.694263,-0.725995,-0.968252,0.0,-0.433320,-0.418878,-0.566253,-0.678022,-0.971486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34997,Q9Y6Y0_336_338_1_1_S338~SLsFEMQQDELIEK,0.213563,0.0,0.749134,-0.062665,0.424471,0.109010,0.033093,0.213563,0.0,...,-0.237642,-0.229769,-0.326227,0.213563,0.0,0.339692,-0.197463,0.293183,0.357531,0.632659
34998,Q9Y6Y0_336_356_1_0~SLSFEMQQDELIEKPMSPMQYAR,-1.000000,0.0,-0.288640,-0.458789,-0.316708,-0.321405,-0.417177,-1.000000,0.0,...,-0.272567,-0.010828,-0.409369,-1.000000,0.0,-0.677587,0.251546,-0.903483,-0.073504,-0.696452
34999,Q9Y6Y0_336_356_1_1_S338~SLsFEMQQDELIEKPMSPMQYAR,0.470627,0.0,0.014271,0.430188,0.132283,0.080787,0.582825,0.470627,0.0,...,0.496345,1.000000,0.353154,0.470627,0.0,0.341903,0.492076,0.097209,0.917837,0.303394
35000,Q9Y6Y8_118_138_1_0~PLTALPFTTGSQDVSNAFSPSISK,-0.153054,0.0,-0.239729,-0.164004,0.423193,0.291979,0.645162,-0.153054,0.0,...,-0.193603,0.050333,-0.288665,-0.153054,0.0,-1.000000,0.300186,0.197979,0.221990,-0.289061


In [85]:
# df_renamed = df_renamed[df_renamed["site"] != "Q15056_21_30_1_1_S24~GSRGsAGGHGSR"]
# df_renamed

,protein_Id,description,protein_name,Peptide,MaxPepProbability,site_start,site_end,n_sites,localized_sites,phospho_sites,...,raw_CV_EGFnINS_2,raw_CV_EGFnINS_5,raw_CV_EGFnINS_10,raw_CV_EGFnINS_15,raw_CV_EGFnINS_90,difference_EGF_vs_EGFnINS,difference_INS_vs_EGFnINS,difference_geometric_mean,min_diff,protein_total_diff
0,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGSGNFGGGR,0.9010,197,199,1,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.211143,0.080023,0.129986,0.080023,4.787742
1,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGsGNFGGGR,1.0000,197,199,1,1,S199,...,47.666831,4.714165,4.992779,17.848973,14.589706,0.627212,0.875666,0.741099,0.627212,4.787742
2,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,GGGGYGGSGDGYNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMK,1.0000,237,271,1,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.040935,0.039200,0.040058,0.039200,4.787742
3,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,GGGGYGGSGDGyNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMK,1.0000,237,271,1,1,Y244,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.130758,0.107351,0.118478,0.107351,4.787742
4,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,1.0000,305,316,1,0,0,...,43.313762,5.630183,10.228770,17.900240,10.319898,0.326720,0.307026,0.316720,0.307026,4.787742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36574,rev_Q96JQ0,0,0,AGAGGPGEARVtLARWPPGAEFDLPR,0.9976,1894,1894,1,1,T1894,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.228274,0.150927,0.185614,0.150927,0.150927
36575,rev_Q9HB15,0,0,sRRPPPR,0.9873,419,419,1,1,S419,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.385817,0.465041,0.423581,0.385817,0.385817
36576,rev_Q9UJW3,0,0,VANQLsGGHVDPITVPEMELFRsAVDLDEK,0.9969,72,89,2,2,S72;S89,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.102486,0.135095,0.117666,0.102486,0.102486
36577,rev_Q9UKR3,0,0,TGACLMNDTIKGPyVQR,0.9834,73,86,1,1,Y86,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.108641,0.253646,0.166001,0.108641,0.108641


In [91]:
df_scaled = pd.merge(df_renamed, subset_scaled, on="site", how="left")
df_scaled

,protein_Id,protein_name,site,raw_abs_EGF_full_r1,raw_abs_EGF_full_r2,raw_abs_EGF_full_r3,raw_abs_EGF_full_r4,raw_abs_EGF_starve_r1,raw_abs_EGF_starve_r2,raw_abs_EGF_starve_r3,...,FC_scaled_INS_5,FC_scaled_INS_10,FC_scaled_INS_90,FC_scaled_EGFnINS_full,FC_scaled_EGFnINS_starve,FC_scaled_EGFnINS_1,FC_scaled_EGFnINS_2,FC_scaled_EGFnINS_5,FC_scaled_EGFnINS_10,FC_scaled_EGFnINS_90
0,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_0~SGSGNFGGGR,0.000000e+00,3.413601e+04,0.000000e+00,0.000000e+00,0.000000,11970.095745,0.000000,...,0.062765,-0.053044,0.392610,1.000000,0.0,0.009989,0.027808,0.150342,0.266855,0.384599
1,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_1_S154~SGsGNFGGGR,2.526291e+06,3.107422e+06,2.844213e+06,2.838165e+06,462877.845884,406460.086218,466606.366000,...,0.179028,0.290848,0.460393,1.000000,0.0,-0.013889,0.076752,0.324961,0.489716,0.582838
2,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_0~NQGGYGGSSSSSSYGSGR,2.538396e+05,2.912180e+05,1.878137e+05,2.302995e+05,370294.907420,351169.823470,391574.055743,...,-0.255660,-0.346432,-0.518686,-0.960321,0.0,-0.298015,-0.083473,-0.121019,-0.236009,-1.000000
3,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S264~NQGGYGGSsSSSSYGSGR,0.000000e+00,3.436628e+04,0.000000e+00,3.184211e+04,0.000000,36947.507319,0.000000,...,-0.101023,-0.349209,-0.490396,-0.873135,0.0,-0.473961,0.472592,-0.426896,1.000000,-0.556594
4,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S266~NQGGYGGSSSsSSYGSGR,4.004880e+05,5.070251e+05,2.948686e+05,0.000000e+00,732858.964004,703950.993495,706990.737660,...,-0.463319,-0.694263,-0.725995,-0.968252,0.0,-0.433320,-0.418878,-0.566253,-0.678022,-0.971486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34997,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_338_1_1_S338~SLsFEMQQDELIEK,3.612031e+04,3.523211e+04,2.328310e+04,2.882678e+04,28158.918559,31337.999945,38445.424548,...,-0.237642,-0.229769,-0.326227,0.213563,0.0,0.339692,-0.197463,0.293183,0.357531,0.632659
34998,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_0~SLSFEMQQDELIEKPMSPMQYAR,0.000000e+00,0.000000e+00,0.000000e+00,1.989511e+04,0.000000,0.000000,0.000000,...,-0.272567,-0.010828,-0.409369,-1.000000,0.0,-0.677587,0.251546,-0.903483,-0.073504,-0.696452
34999,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_1_S338~SLsFEMQQDELIEKPMSPMQYAR,2.252726e+05,2.548493e+05,1.697470e+05,1.780225e+05,202512.144807,165807.326109,237987.604392,...,0.496345,1.000000,0.353154,0.470627,0.0,0.341903,0.492076,0.097209,0.917837,0.303394
35000,Q9Y6Y8,SEC23IP,Q9Y6Y8_118_138_1_0~PLTALPFTTGSQDVSNAFSPSISK,0.000000e+00,0.000000e+00,3.689021e+03,0.000000e+00,0.000000,0.000000,5253.875296,...,-0.193603,0.050333,-0.288665,-0.153054,0.0,-1.000000,0.300186,0.197979,0.221990,-0.289061


In [92]:
df_scaled.to_excel("Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff_scaled.xlsx", index=False)

# Save data as a "tsv" for faster loading

In [2]:
df = pd.read_excel("Experiment/1_HEK293T/Data/Processed/Full_dataset_HEK293T_functional_names_diff_scaled_phPlus.xlsx")

In [3]:
df.to_csv("Experiment/1_HEK293T/Data/Processed/Full_dataset_HEK293T_functional_names_diff_scaled_phPlus.tsv", sep="\t", index=False)

In [7]:
new_df = pd.read_csv("Experiment/1_HEK293T/Data/Processed/Full_dataset_HEK293T_functional_names_diff_scaled_phPlus.tsv", sep="\t")

In [25]:
from pathlib import Path
path = "Experiment/1_hTERT_HME1/Data/Processed/"
folder = Path(path)

for file in folder.iterdir():
    if ".xlsx" in file.name and "Full_dataset" in file.name:
        name = str(file)
        df = pd.read_excel(name)

        tsv_file = name.replace(".xlsx", ".tsv")
        df.to_csv(tsv_file, sep="\t", index=False)

        print(f"saved file {tsv_file}")

saved file Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff.tsv
saved file Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_diff.tsv
saved file Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff_scaled_phPlus.tsv
saved file Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1.tsv
saved file Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff_scaled_phPlus_localized.tsv
saved file Experiment/1_hTERT_HME1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff_scaled.tsv
